# 🚀 Interactive Enterprise RAG Architecture

## Complete 8-Agent Workflow with Human-in-the-Loop

This notebook provides a **fully interactive** interface to:
1. ✅ **Upload or select** an RFP file
2. ✅ **Automatically check** if database is indexed
3. ✅ **Generate proposal** using 8-agent system
4. ✅ **Review and approve** at each stage

---

## 📋 Prerequisites

Before starting:
- ✅ OpenAI API key in `.env` file
- ✅ Historical proposals in `data/proposals/` folder
- ✅ Run this notebook from top to bottom

**First time users**: The notebook will guide you through database indexing.

## Step 1: Setup and Initialization

In [ ]:
# Import libraries
import sys
import os
from pathlib import Path
import shutil
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# Add src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))

# Load environment
from dotenv import load_dotenv
load_dotenv(project_root / '.env')

# Check API key
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    display(HTML('<div style="background-color: #ffcccc; padding: 15px; border-radius: 5px;">' +
                 '<h3>❌ ERROR: OPENAI_API_KEY not found</h3>' +
                 '<p>Please add your OpenAI API key to the .env file:</p>' +
                 '<pre>OPENAI_API_KEY=sk-your-actual-key-here</pre>' +
                 '<p>Get your API key from: <a href="https://platform.openai.com/api-keys" target="_blank">OpenAI Platform</a></p>' +
                 '</div>'))
    raise ValueError("OpenAI API key not found")
else:
    display(HTML(f'<div style="background-color: #ccffcc; padding: 10px; border-radius: 5px;">' +
                 f'<b>✅ OpenAI API key loaded:</b> {api_key[:15]}...</div>'))

# Define paths
data_dir = project_root / 'data'
rfp_dir = data_dir / 'rfps'
proposal_dir = data_dir / 'proposals'
output_dir = project_root / 'output'
db_path = data_dir / 'chroma_db'

# Create directories if they don't exist
rfp_dir.mkdir(parents=True, exist_ok=True)
proposal_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

print("\n✅ Setup complete!")

## Step 2: Check Database Status

In [ ]:
# Check if database is indexed
database_indexed = db_path.exists()

if not database_indexed:
    display(HTML('<div style="background-color: #fff3cd; padding: 15px; border-radius: 5px; border: 2px solid #ffc107;">' +
                 '<h3>⚠️ Database Not Indexed</h3>' +
                 '<p>Your proposal database needs to be indexed first.</p>' +
                 '<p><b>What this means:</b> The system needs to process your historical proposals ' +
                 'to learn patterns and generate better content.</p>' +
                 '<p><b>What to do:</b></p>' +
                 '<ol>' +
                 '<li>Make sure you have proposal files (.pptx) in <code>data/proposals/</code></li>' +
                 '<li>Run the indexing cell below</li>' +
                 '</ol>' +
                 '</div>'))
else:
    # Check database stats
    from rag_engine import RAGEngine
    rag_temp = RAGEngine(api_key=api_key, db_path=str(db_path), verbose=False)
    stats = rag_temp.get_collection_stats()
    
    total_chunks = stats.get('total_chunks', 0)
    rfp_chunks = stats.get('doc_types', {}).get('rfp', 0)
    proposal_chunks = stats.get('doc_types', {}).get('proposal', 0)
    
    if total_chunks == 0:
        display(HTML('<div style="background-color: #fff3cd; padding: 15px; border-radius: 5px; border: 2px solid #ffc107;">' +
                     '<h3>⚠️ Database is Empty</h3>' +
                     '<p>The database exists but contains no data.</p>' +
                     '<p>Please run the indexing cell below.</p>' +
                     '</div>'))
        database_indexed = False
    else:
        display(HTML(f'<div style="background-color: #d4edda; padding: 15px; border-radius: 5px; border: 2px solid #28a745;">' +
                     f'<h3>✅ Database Ready</h3>' +
                     f'<p><b>Total chunks:</b> {total_chunks}</p>' +
                     f'<p><b>RFP chunks:</b> {rfp_chunks}</p>' +
                     f'<p><b>Proposal chunks:</b> {proposal_chunks}</p>' +
                     f'<p>The system is ready to generate proposals!</p>' +
                     f'</div>'))

## Step 3: Index Database (Run if Needed)

⚠️ **Only run this cell if the check above says database is not indexed**

This will take 2-5 minutes depending on the number of files.

In [ ]:
# Index database (only run if needed)
if not database_indexed:
    display(HTML('<div style="background-color: #cce5ff; padding: 15px; border-radius: 5px;">' +
                 '<h3>🔄 Indexing Database...</h3>' +
                 '<p>This may take 2-5 minutes. Please wait...</p>' +
                 '</div>'))
    
    # Check for files
    rfp_files = list(rfp_dir.glob('*.pdf'))
    proposal_files = list(proposal_dir.glob('*.pptx'))
    
    print(f"\n📁 Found:")
    print(f"   RFP files: {len(rfp_files)}")
    print(f"   Proposal files: {len(proposal_files)}")
    
    if len(proposal_files) == 0:
        display(HTML('<div style="background-color: #ffcccc; padding: 15px; border-radius: 5px;">' +
                     '<h3>❌ No Proposal Files Found</h3>' +
                     '<p>Please add at least 2-3 historical proposal files (.pptx) to:</p>' +
                     '<pre>data/proposals/</pre>' +
                     '<p>Then restart this notebook.</p>' +
                     '</div>'))
    else:
        from rag_engine import RAGEngine
        
        # Initialize RAG engine
        rag = RAGEngine(api_key=api_key, db_path=str(db_path), verbose=True)
        
        # Index documents
        rag.index_documents(
            rfp_dir=str(rfp_dir),
            proposal_dir=str(proposal_dir)
        )
        
        # Show stats
        stats = rag.get_collection_stats()
        
        display(HTML(f'<div style="background-color: #d4edda; padding: 15px; border-radius: 5px; border: 2px solid #28a745;">' +
                     f'<h3>✅ Indexing Complete!</h3>' +
                     f'<p><b>Total chunks indexed:</b> {stats.get("total_chunks", 0)}</p>' +
                     f'<p><b>RFP chunks:</b> {stats.get("doc_types", {}).get("rfp", 0)}</p>' +
                     f'<p><b>Proposal chunks:</b> {stats.get("doc_types", {}).get("proposal", 0)}</p>' +
                     f'<p>You can now proceed to select your RFP file!</p>' +
                     f'</div>'))
        
        database_indexed = True
else:
    print("✅ Database already indexed. Skipping this step.")

## Step 4: Select or Upload RFP File

Choose how you want to provide your RFP:
- **Option A**: Upload a NEW RFP file from your computer
- **Option B**: Select an EXISTING RFP from the data/rfps/ folder

In [ ]:
# Initialize variables
selected_rfp_path = None
rfp_selection_method = None

# Create selection radio buttons
selection_method = widgets.RadioButtons(
    options=['Upload NEW RFP', 'Select EXISTING RFP'],
    description='Choose:',
    disabled=False
)

# Create file upload widget
file_upload = widgets.FileUpload(
    accept='.pdf',
    multiple=False,
    description='Upload RFP (PDF)'
)

# Create dropdown for existing files
existing_rfp_files = list(rfp_dir.glob('*.pdf'))
if existing_rfp_files:
    rfp_dropdown = widgets.Dropdown(
        options=[f.name for f in existing_rfp_files],
        description='Select RFP:',
        disabled=False
    )
else:
    rfp_dropdown = widgets.HTML(
        value='<p style="color: orange;">⚠️ No RFP files found in data/rfps/</p>'
    )

# Create confirm button
confirm_button = widgets.Button(
    description='Confirm Selection',
    button_style='success',
    icon='check'
)

# Create output area
output_area = widgets.Output()

# Create container for widgets
upload_container = widgets.VBox([file_upload], layout=widgets.Layout(display='none'))
dropdown_container = widgets.VBox([rfp_dropdown], layout=widgets.Layout(display='none'))

# Handler for selection method change
def on_method_change(change):
    if change['new'] == 'Upload NEW RFP':
        upload_container.layout.display = 'block'
        dropdown_container.layout.display = 'none'
    else:
        upload_container.layout.display = 'none'
        dropdown_container.layout.display = 'block'

selection_method.observe(on_method_change, names='value')

# Handler for confirm button
def on_confirm_click(b):
    global selected_rfp_path, rfp_selection_method
    
    with output_area:
        clear_output()
        
        if selection_method.value == 'Upload NEW RFP':
            # Handle file upload
            if len(file_upload.value) == 0:
                display(HTML('<p style="color: red;">❌ Please upload a PDF file first!</p>'))
                return
            
            # Save uploaded file
            uploaded_file = list(file_upload.value.values())[0]
            file_name = list(file_upload.value.keys())[0]
            
            # Save to rfps directory
            target_path = rfp_dir / file_name
            with open(target_path, 'wb') as f:
                f.write(uploaded_file['content'])
            
            selected_rfp_path = str(target_path)
            rfp_selection_method = 'upload'
            
            display(HTML(f'<div style="background-color: #d4edda; padding: 10px; border-radius: 5px;">' +
                         f'<b>✅ RFP Uploaded Successfully!</b><br>' +
                         f'File: {file_name}<br>' +
                         f'Saved to: {target_path}<br>' +
                         f'<br>You can now proceed to the next step.' +
                         f'</div>'))
        else:
            # Handle existing file selection
            if isinstance(rfp_dropdown, widgets.HTML):
                display(HTML('<p style="color: red;">❌ No existing RFP files available. Please upload a file instead.</p>'))
                return
            
            selected_file = rfp_dropdown.value
            selected_rfp_path = str(rfp_dir / selected_file)
            rfp_selection_method = 'existing'
            
            display(HTML(f'<div style="background-color: #d4edda; padding: 10px; border-radius: 5px;">' +
                         f'<b>✅ RFP Selected!</b><br>' +
                         f'File: {selected_file}<br>' +
                         f'Path: {selected_rfp_path}<br>' +
                         f'<br>You can now proceed to the next step.' +
                         f'</div>'))

confirm_button.on_click(on_confirm_click)

# Display UI
display(HTML('<h3>📄 Select Your RFP File</h3>'))
display(selection_method)
display(upload_container)
display(dropdown_container)
display(confirm_button)
display(output_area)

# Trigger initial display
if selection_method.value == 'Upload NEW RFP':
    upload_container.layout.display = 'block'
else:
    dropdown_container.layout.display = 'block'

## Step 5: Parse RFP

Extract text and metadata from the selected RFP file.

In [ ]:
# Check if RFP was selected
if selected_rfp_path is None:
    display(HTML('<div style="background-color: #ffcccc; padding: 15px; border-radius: 5px;">' +
                 '<h3>❌ No RFP Selected</h3>' +
                 '<p>Please go back to Step 4 and select or upload an RFP file.</p>' +
                 '</div>'))
else:
    from rfp_parser import parse_rfp
    
    display(HTML('<h3>🔄 Parsing RFP...</h3>'))
    
    try:
        rfp_data = parse_rfp(selected_rfp_path, verbose=True)
        rfp_text = rfp_data['full_text']
        project_name = rfp_data['project_name']
        
        display(HTML(f'<div style="background-color: #d4edda; padding: 15px; border-radius: 5px; border: 2px solid #28a745;">' +
                     f'<h3>✅ RFP Parsed Successfully!</h3>' +
                     f'<p><b>Project:</b> {project_name}</p>' +
                     f'<p><b>Pages:</b> {rfp_data["page_count"]}</p>' +
                     f'<p><b>Text length:</b> {len(rfp_text):,} characters</p>' +
                     f'<br>' +
                     f'<details>' +
                     f'<summary><b>Click to view first 500 characters</b></summary>' +
                     f'<pre style="background-color: #f8f9fa; padding: 10px; margin-top: 10px;">{rfp_text[:500]}...</pre>' +
                     f'</details>' +
                     f'</div>'))
    except Exception as e:
        display(HTML(f'<div style="background-color: #ffcccc; padding: 15px; border-radius: 5px;">' +
                     f'<h3>❌ Error Parsing RFP</h3>' +
                     f'<p>{str(e)}</p>' +
                     f'<p>Please make sure the file is a valid PDF.</p>' +
                     f'</div>'))
        rfp_text = None
        project_name = None

## Step 6: Initialize RAG Engine and Agents

Set up the 8-agent system.

In [ ]:
if not database_indexed:
    display(HTML('<div style="background-color: #ffcccc; padding: 15px; border-radius: 5px;">' +
                 '<h3>❌ Database Not Indexed</h3>' +
                 '<p>Please go back to Step 3 and index the database first.</p>' +
                 '</div>'))
elif rfp_text is None:
    display(HTML('<div style="background-color: #ffcccc; padding: 15px; border-radius: 5px;">' +
                 '<h3>❌ No RFP Text Available</h3>' +
                 '<p>Please go back to Step 5 and parse the RFP successfully.</p>' +
                 '</div>'))
else:
    from rag_engine import RAGEngine
    from langchain_openai import ChatOpenAI
    from content_generator import ContentGenerator
    
    display(HTML('<h3>🔄 Initializing 8-Agent System...</h3>'))
    
    # Initialize RAG engine
    print("\n📚 Loading RAG Engine...")
    rag = RAGEngine(
        api_key=api_key,
        db_path=str(db_path),
        verbose=False
    )
    
    # Initialize LLM
    print("🤖 Initializing LLM...")
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        api_key=api_key,
        temperature=0.3
    )
    
    # Initialize Content Generator (Agents 3-6)
    print("🎯 Initializing Content Generator (Agents 3-6)...")
    content_gen = ContentGenerator(
        rag_engine=rag,
        llm=llm,
        verbose=True
    )
    
    display(HTML('<div style="background-color: #d4edda; padding: 15px; border-radius: 5px; border: 2px solid #28a745;">' +
                 '<h3>✅ System Initialized!</h3>' +
                 '<p>All 8 agents are ready to generate your proposal.</p>' +
                 '<ul>' +
                 '<li>✅ RAG Engine loaded</li>' +
                 '<li>✅ LLM initialized (gpt-4o-mini)</li>' +
                 '<li>✅ Agent 3: WorkstreamAgent</li>' +
                 '<li>✅ Agent 4: DependencyAgent</li>' +
                 '<li>✅ Agent 5: ModuleAgent</li>' +
                 '<li>✅ Agent 6: TimelineOptimizer</li>' +
                 '</ul>' +
                 '</div>'))

## Step 7: Generate Content (Run Agents 3-6)

This will:
1. Extract workstreams from RFP and learn from historical proposals
2. Identify dependencies between workstreams
3. Build detailed modules with activities
4. Optimize the timeline

**This takes 5-10 minutes** depending on RFP size and complexity.

In [ ]:
if rfp_text is None or project_name is None:
    display(HTML('<div style="background-color: #ffcccc; padding: 15px; border-radius: 5px;">' +
                 '<h3>❌ Cannot Generate Content</h3>' +
                 '<p>Please complete all previous steps first.</p>' +
                 '</div>'))
else:
    display(HTML('<div style="background-color: #cce5ff; padding: 15px; border-radius: 5px;">' +
                 '<h3>🚀 Starting Content Generation...</h3>' +
                 '<p>This will take 5-10 minutes. Watch the progress below.</p>' +
                 '<p><b>Agents running:</b></p>' +
                 '<ul>' +
                 '<li>Agent 3: Extracting workstreams from RFP and learning from repository</li>' +
                 '<li>Agent 4: Analyzing dependencies</li>' +
                 '<li>Agent 5: Building MECE modules with activities</li>' +
                 '<li>Agent 6: Optimizing timeline</li>' +
                 '</ul>' +
                 '</div>'))
    
    try:
        # Generate content
        project_structure = content_gen.generate_content(
            rfp_text=rfp_text,
            project_name=project_name,
            duration_weeks=None  # Auto-detect from RFP
        )
        
        # Show summary
        display(HTML(f'<div style="background-color: #d4edda; padding: 15px; border-radius: 5px; border: 2px solid #28a745;">' +
                     f'<h3>✅ Content Generation Complete!</h3>' +
                     f'<p><b>Project:</b> {project_structure.project_name}</p>' +
                     f'<p><b>Workstreams:</b> {len(project_structure.workstreams)}</p>' +
                     f'<p><b>Total Modules:</b> {sum(len(ws.modules) for ws in project_structure.workstreams)}</p>' +
                     f'<p><b>Total Activities:</b> {sum(len(mod.activities) for ws in project_structure.workstreams for mod in ws.modules)}</p>' +
                     f'<p><b>Dependencies:</b> {len(project_structure.dependencies.workstream_dependencies)}</p>' +
                     f'<br>' +
                     f'<p><b>Workstream Breakdown:</b></p>' +
                     f'<ul>' +
                     f'{"".join([f"<li>{ws.name}: {len(ws.modules)} modules, {ws.duration_weeks} weeks</li>" for ws in project_structure.workstreams])}' +
                     f'</ul>' +
                     f'</div>'))
        
        # Store for next steps
        generated_content = project_structure
        
    except Exception as e:
        display(HTML(f'<div style="background-color: #ffcccc; padding: 15px; border-radius: 5px;">' +
                     f'<h3>❌ Error During Content Generation</h3>' +
                     f'<p><b>Error:</b> {str(e)}</p>' +
                     f'<p>Please check the error message and try again.</p>' +
                     f'</div>'))
        import traceback
        print("\n" + "="*80)
        print("FULL ERROR TRACEBACK:")
        print("="*80)
        traceback.print_exc()
        generated_content = None

## Step 8: Review Generated Structure

Review the complete project structure before saving.

In [ ]:
if 'generated_content' not in locals() or generated_content is None:
    display(HTML('<div style="background-color: #ffcccc; padding: 15px; border-radius: 5px;">' +
                 '<h3>❌ No Content Generated</h3>' +
                 '<p>Please run Step 7 first to generate content.</p>' +
                 '</div>'))
else:
    print("\n" + "="*80)
    print("📊 GENERATED PROJECT STRUCTURE")
    print("="*80)
    print(f"\nProject: {generated_content.project_name}")
    print(f"\nWorkstreams: {len(generated_content.workstreams)}")
    
    for i, ws in enumerate(generated_content.workstreams, 1):
        print(f"\n{'='*80}")
        print(f"{i}. {ws.name}")
        print(f"{'='*80}")
        print(f"   Duration: {ws.duration_weeks} weeks (Week {ws.start_week}-{ws.end_week})")
        print(f"   Modules: {len(ws.modules)}")
        print(f"   Confidence: {ws.confidence_score:.2f}")
        print(f"\n   Rationale: {ws.rationale}")
        
        if ws.evidence_sources:
            print(f"\n   📚 Evidence sources:")
            for source in ws.evidence_sources[:3]:
                print(f"      • {source}")
        
        print(f"\n   📦 Modules:")
        for j, mod in enumerate(ws.modules, 1):
            print(f"\n      {j}. {mod.name}")
            print(f"         Duration: {mod.duration_weeks} weeks (Week {mod.start_week}-{mod.end_week})")
            print(f"         Activities: {len(mod.activities)}")
            print(f"         Deliverables: {len(mod.deliverables)}")
            
            if mod.deliverables:
                print(f"         Key deliverables:")
                for deliv in mod.deliverables[:3]:
                    print(f"            • {deliv}")
    
    print(f"\n{'='*80}")
    print("\n📊 SUMMARY")
    print("="*80)
    print(f"Total Workstreams: {len(generated_content.workstreams)}")
    print(f"Total Modules: {sum(len(ws.modules) for ws in generated_content.workstreams)}")
    print(f"Total Activities: {sum(len(mod.activities) for ws in generated_content.workstreams for mod in ws.modules)}")
    print(f"Total Deliverables: {sum(len(mod.deliverables) for ws in generated_content.workstreams for mod in ws.modules)}")
    print(f"\nProject Duration: {max(ws.end_week for ws in generated_content.workstreams)} weeks")
    
    display(HTML('<div style="background-color: #d4edda; padding: 15px; border-radius: 5px; margin-top: 20px;">' +
                 '<h3>✅ Structure Generated Successfully!</h3>' +
                 '<p>Review the structure above. If you\'re satisfied, proceed to the next step to save the output.</p>' +
                 '</div>'))

## Step 9: Save Project (Optional)

Save the generated project structure to JSON for future reference.

In [ ]:
if 'generated_content' not in locals() or generated_content is None:
    display(HTML('<div style="background-color: #ffcccc; padding: 15px; border-radius: 5px;">' +
                 '<h3>❌ No Content to Save</h3>' +
                 '<p>Please generate content first.</p>' +
                 '</div>'))
else:
    import json
    from datetime import datetime
    
    # Create filename
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = output_dir / f"{project_name}_{timestamp}.json"
    
    # Convert to dict (you may need to implement a to_dict() method)
    # For now, just save basic info
    project_data = {
        "project_name": generated_content.project_name,
        "generated_at": timestamp,
        "workstreams": [
            {
                "name": ws.name,
                "duration_weeks": ws.duration_weeks,
                "start_week": ws.start_week,
                "rationale": ws.rationale,
                "modules": [
                    {
                        "name": mod.name,
                        "duration_weeks": mod.duration_weeks,
                        "deliverables": mod.deliverables,
                        "activities_count": len(mod.activities)
                    }
                    for mod in ws.modules
                ]
            }
            for ws in generated_content.workstreams
        ],
        "summary": {
            "total_workstreams": len(generated_content.workstreams),
            "total_modules": sum(len(ws.modules) for ws in generated_content.workstreams),
            "project_duration_weeks": max(ws.end_week for ws in generated_content.workstreams)
        }
    }
    
    # Save to JSON
    with open(output_file, 'w') as f:
        json.dump(project_data, f, indent=2)
    
    display(HTML(f'<div style="background-color: #d4edda; padding: 15px; border-radius: 5px; border: 2px solid #28a745;">' +
                 f'<h3>✅ Project Saved!</h3>' +
                 f'<p><b>File:</b> {output_file.name}</p>' +
                 f'<p><b>Location:</b> {output_file}</p>' +
                 f'</div>'))

## 🎉 Workflow Complete!

You have successfully:
1. ✅ Selected or uploaded an RFP
2. ✅ Parsed the RFP content
3. ✅ Generated proposal structure using 8-agent system
4. ✅ Reviewed the generated content
5. ✅ Saved the project

---

### Next Steps

The generated structure includes:
- **Workstreams** with durations and rationale
- **Modules** with deliverables and activities
- **Dependencies** between workstreams
- **Evidence sources** from historical proposals

You can now use this structure to:
- Create a PowerPoint presentation (coming soon in full workflow)
- Export to Excel for timeline planning
- Share with your team for review

---

### Questions or Issues?

Check the [QUICK_START.md](../QUICK_START.md) or create a GitHub issue.